# 04 - Direct Model API Calls

If you deploy independently, populate `env/.env` with `APIM_BASE_URL`, `APIM_MODEL_NAME` and `APIM_TOKEN_SCOPE` (or `APIM_API_KEY`) for APIM, `FOUNDRY_BASE_URL`, `FOUNDRY_MODEL_NAME` and `FOUNDRY_TOKEN_SCOPE` (or `FOUNDRY_API_KEY`) for Foundry, and `AZURE_AI_PROJECT_ENDPOINT` for the Azure AI Projects SDK; if deployed through this demo's notebooks, the endpoint/model/scope fields are populated for you.

Call a model through APIM or Foundry using the OpenAI SDK, HTTPS, or Azure AI Projects SDK. Use **Python (byom01-capability-testing)** and the `notebooks/` working directory.

Run the settings and authentication cells for your chosen endpoint, then select one request example. Each execution makes one paid Responses request. Model output follows the endpoint's storage and retention policy.

## Load Endpoint Settings

Load APIM settings from `env/.env`. Use the complete gateway base URL ending in `/openai/v1/`, without `/responses`, and the deployment name or alias accepted by the gateway.

In [ ]:
from pathlib import Path
from urllib.parse import urlsplit

from dotenv import dotenv_values

demo_root = Path("..").resolve()
env_path = demo_root / "env" / ".env"
settings = dotenv_values(env_path)
apim_base_url = (settings.get("APIM_BASE_URL") or "").strip().rstrip("/") + "/"
apim_model = (settings.get("APIM_MODEL_NAME") or "").strip()
endpoint = urlsplit(apim_base_url)
if endpoint.scheme != "https" or not endpoint.netloc or endpoint.query or endpoint.fragment or endpoint.username:
    raise ValueError("Set APIM_BASE_URL in env/.env to an HTTPS OpenAI v1 base URL without credentials, query, or fragment.")
if not apim_model:
    raise ValueError("Set APIM_MODEL_NAME in env/.env to the gateway's deployment name or alias.")
apim_api_key = (settings.get("APIM_API_KEY") or "").strip()
apim_key_header = (settings.get("APIM_API_KEY_HEADER") or "").strip()
apim_token_scope = (settings.get("APIM_TOKEN_SCOPE") or "").strip()
if not apim_api_key and not apim_token_scope:
    raise ValueError("Set APIM_TOKEN_SCOPE in env/.env for Entra authentication, or supply an API key accepted by your gateway.")
poem_prompt = "Write an original short poem about the sea at sunrise."
print(f"APIM endpoint: {apim_base_url}")
print(f"Model: {apim_model}")

## Configure Authentication

Use `APIM_API_KEY` when set; otherwise use DefaultAzureCredential with `APIM_TOKEN_SCOPE`. The selected identity must be authorized at the endpoint. Each request prints the chosen method without exposing credentials.

Set `APIM_API_KEY_HEADER` if the gateway expects a header such as `Ocp-Apim-Subscription-Key` or `api-key`; leave it empty for bearer-key authentication. Any separate JWT requirements still apply.

In [ ]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

auth_method = "API key" if apim_api_key else "Microsoft Entra ID (DefaultAzureCredential)"
auth = apim_api_key or get_bearer_token_provider(DefaultAzureCredential(), apim_token_scope)

## APIM via SDK

Create an OpenAI client with the APIM base URL and send a Responses request.

In [ ]:
from openai import OpenAI, omit

print(f"Using {auth_method}")
apim_client = OpenAI(
    base_url=apim_base_url,
    api_key=auth,
    default_headers={apim_key_header: apim_api_key} if apim_api_key and apim_key_header else {},
    max_retries=0,
    timeout=60.0,
)
response = apim_client.responses.create(
    model=apim_model,
    input=poem_prompt,
    extra_headers={"Authorization": omit} if apim_api_key and apim_key_header else {},
)
assert response.status == "completed", f"Response did not complete: {response.status}"
assert response.output_text.strip(), "Model returned no output text"
print(response.output_text)
print(f"Response ID: {response.id}")

## APIM via HTTPS

Send the Responses request as an HTTP POST with an explicit URL, headers, and JSON body.

In [ ]:
import httpx

print(f"Using {auth_method}")
headers = {"Content-Type": "application/json"}
if apim_api_key and apim_key_header:
    headers[apim_key_header] = apim_api_key
else:
    bearer = auth() if callable(auth) else auth
    headers["Authorization"] = f"Bearer {bearer}"

with httpx.Client(transport=httpx.HTTPTransport(retries=0), timeout=60.0, follow_redirects=False) as http_client:
    http_response = http_client.post(
        apim_base_url + "responses",
        headers=headers,
        json={"model": apim_model, "input": poem_prompt},
    )
    http_response.raise_for_status()
    result = http_response.json()
assert result["status"] == "completed", f"Response did not complete: {result['status']}"
poem = "\n".join(
    content["text"]
    for item in result["output"] if item["type"] == "message"
    for content in item["content"] if content["type"] == "output_text"
)
assert poem.strip(), "Model returned no output text"
print(poem)
print(f"Response ID: {result['id']}")

## Load Foundry Settings

Load Foundry settings from `env/.env`. A resource base URL has the form `https://<resource>.services.ai.azure.com/openai/v1/`; a project base includes `/api/projects/<project>/openai/v1/`. Use the exact model reference accepted by the endpoint, such as `byom01-gateway/gpt-5.4`.

For the Azure AI Projects SDK example, also provide `AZURE_AI_PROJECT_ENDPOINT` without `/openai/v1/`.

In [ ]:
from pathlib import Path
from urllib.parse import urlsplit

from dotenv import dotenv_values

foundry_settings = dotenv_values(Path("..").resolve() / "env" / ".env")
foundry_base_url = (foundry_settings.get("FOUNDRY_BASE_URL") or "").strip().rstrip("/") + "/"
foundry_model = (foundry_settings.get("FOUNDRY_MODEL_NAME") or "").strip()
foundry_endpoint = urlsplit(foundry_base_url)
if foundry_endpoint.scheme != "https" or not foundry_endpoint.netloc or foundry_endpoint.query or foundry_endpoint.fragment or foundry_endpoint.username:
    raise ValueError("Set FOUNDRY_BASE_URL in env/.env to an HTTPS OpenAI v1 base URL without credentials, query, or fragment.")
if not foundry_model:
    raise ValueError("Set FOUNDRY_MODEL_NAME in env/.env to the exact model reference to test.")
foundry_api_key = (foundry_settings.get("FOUNDRY_API_KEY") or "").strip()
foundry_key_header = (foundry_settings.get("FOUNDRY_API_KEY_HEADER") or "").strip()
foundry_token_scope = (foundry_settings.get("FOUNDRY_TOKEN_SCOPE") or "").strip()
if not foundry_api_key and not foundry_token_scope:
    raise ValueError("Set FOUNDRY_TOKEN_SCOPE in env/.env for Entra authentication, or supply an API key accepted by your endpoint.")
foundry_prompt = "Write an original short poem about the sea at sunrise."
print(f"Foundry endpoint: {foundry_base_url}")
print(f"Model: {foundry_model}")

## Configure Foundry Authentication

Use `FOUNDRY_API_KEY` when set; otherwise use DefaultAzureCredential with `FOUNDRY_TOKEN_SCOPE` (`https://ai.azure.com/.default` for Foundry v1). Set `FOUNDRY_API_KEY_HEADER` if a custom key header is required. Each request prints the chosen method without exposing credentials.

In [ ]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

foundry_auth_method = "API key" if foundry_api_key else "Microsoft Entra ID (DefaultAzureCredential)"
foundry_auth = foundry_api_key or get_bearer_token_provider(DefaultAzureCredential(), foundry_token_scope)

## Foundry via OpenAI SDK

Create an OpenAI client with the Foundry base URL and send a Responses request using the model name.

In [ ]:
from openai import OpenAI, omit

print(f"Using {foundry_auth_method}")
foundry_client = OpenAI(
    base_url=foundry_base_url,
    api_key=foundry_auth,
    default_headers={foundry_key_header: foundry_api_key} if foundry_api_key and foundry_key_header else {},
    max_retries=0,
    timeout=60.0,
)
foundry_response = foundry_client.responses.create(
    model=foundry_model,
    input=foundry_prompt,
    extra_headers={"Authorization": omit} if foundry_api_key and foundry_key_header else {},
)
assert foundry_response.status == "completed", f"Response did not complete: {foundry_response.status}"
assert foundry_response.output_text.strip(), "Model returned no output text"
print(foundry_response.output_text)
print(f"Response ID: {foundry_response.id}")

## Foundry via HTTPS

Send the same Responses request as an HTTP POST using explicit headers and JSON.

In [ ]:
import httpx

print(f"Using {foundry_auth_method}")
foundry_headers = {"Content-Type": "application/json"}
if foundry_api_key and foundry_key_header:
    foundry_headers[foundry_key_header] = foundry_api_key
else:
    foundry_bearer = foundry_auth() if callable(foundry_auth) else foundry_auth
    foundry_headers["Authorization"] = f"Bearer {foundry_bearer}"

with httpx.Client(transport=httpx.HTTPTransport(retries=0), timeout=60.0, follow_redirects=False) as foundry_http_client:
    foundry_http_response = foundry_http_client.post(
        foundry_base_url + "responses",
        headers=foundry_headers,
        json={"model": foundry_model, "input": foundry_prompt},
    )
    foundry_http_response.raise_for_status()
    foundry_result = foundry_http_response.json()
assert foundry_result["status"] == "completed", f"Response did not complete: {foundry_result['status']}"
foundry_poem = "\n".join(
    content["text"]
    for item in foundry_result["output"] if item["type"] == "message"
    for content in item["content"] if content["type"] == "output_text"
)
assert foundry_poem.strip(), "Model returned no output text"
print(foundry_poem)
print(f"Response ID: {foundry_result['id']}")

## Foundry via Azure AI Projects SDK

Create AIProjectClient with `AZURE_AI_PROJECT_ENDPOINT`, obtain an OpenAI client through `get_openai_client()`, and send a Responses request with the same model and prompt.

AIProjectClient requires a TokenCredential. The `api_key` argument to `get_openai_client()` selects the configured key or Entra token provider for inference; project-management operations still require Entra.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from openai import omit

foundry_project_endpoint = (foundry_settings.get("AZURE_AI_PROJECT_ENDPOINT") or "").strip().rstrip("/")
if not foundry_project_endpoint:
    raise ValueError("Set AZURE_AI_PROJECT_ENDPOINT in env/.env to the consuming Foundry project endpoint, without /openai/v1/.")
print(f"Using {foundry_auth_method}")
foundry_project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)
foundry_project_openai_client = foundry_project_client.get_openai_client(
    api_key=foundry_auth,
    default_headers={foundry_key_header: foundry_api_key} if foundry_api_key and foundry_key_header else {},
    max_retries=0,
    timeout=60.0,
)
foundry_project_response = foundry_project_openai_client.responses.create(
    model=foundry_model,
    input=foundry_prompt,
    extra_headers={"Authorization": omit} if foundry_api_key and foundry_key_header else {},
)
assert foundry_project_response.status == "completed", f"Response did not complete: {foundry_project_response.status}"
assert foundry_project_response.output_text.strip(), "Model returned no output text"
print(foundry_project_response.output_text)
print(f"Response ID: {foundry_project_response.id}")